# Direct REST API call
> No SDK module required, access tokens and pagination required

##### Function for obtaining access token via certificate thumbprint

In [ ]:
$config = Get-Content "C:\vs\config\clientconfiguration.json" | ConvertFrom-Json
$Global:AccessToken = $null
$Global:TokenExpiry = Get-Date

function Get-AccessToken {
    if ($Global:AccessToken -and (Get-Date) -lt $Global:TokenExpiry) { return $Global:AccessToken }
    $cert = Get-ChildItem Cert:\CurrentUser\My | Where-Object Thumbprint -eq $config.Thumbprint
    if (-not $cert) { throw "Cert not found" }
    if (-not $cert.HasPrivateKey) { throw "Cert has no private key" }
    $b64u = { param($bytes)
        ($b = [Convert]::ToBase64String($bytes)).TrimEnd('=').Replace('+', '-').Replace('/', '_')
    }
    $h = @{ alg = "RS256"; typ = "JWT"; x5t = & $b64u ($cert.GetCertHash()) }
    $now = [DateTimeOffset]::UtcNow
    $p = @{
        aud = "https://login.microsoftonline.com/$($config.TenantId)/oauth2/v2.0/token"
        iss = $config.ClientId; sub = $config.ClientId
        jti = [guid]::NewGuid().ToString()
        nbf = [int]$now.ToUnixTimeSeconds()
        exp = [int]$now.AddMinutes(10).ToUnixTimeSeconds()
    }
    $s1 = & $b64u ([Text.Encoding]::UTF8.GetBytes((ConvertTo-Json $h -Compress)))
    $s2 = & $b64u ([Text.Encoding]::UTF8.GetBytes((ConvertTo-Json $p -Compress)))
    $toSign = [Text.Encoding]::UTF8.GetBytes("$s1.$s2")
    $rsa = [System.Security.Cryptography.X509Certificates.RSACertificateExtensions]::GetRSAPrivateKey($cert)
    if (-not $rsa) { $rsa = $cert.PrivateKey }
    $sigBytes =
    if ($rsa -is [System.Security.Cryptography.RSA]) {
        $rsa.SignData($toSign, [System.Security.Cryptography.HashAlgorithmName]::SHA256, [System.Security.Cryptography.RSASignaturePadding]::Pkcs1)
    }
    else {
        $rsa.SignData($toSign, 'SHA256')
    }
    $assert = "$s1.$s2." + (& $b64u $sigBytes)
    $body = @{
        client_id             = $config.ClientId
        scope                 = "https://graph.microsoft.com/.default"
        grant_type            = "client_credentials"
        client_assertion_type = "urn:ietf:params:oauth:client-assertion-type:jwt-bearer"
        client_assertion      = $assert
    }
    $url = "https://login.microsoftonline.com/$($config.TenantId)/oauth2/v2.0/token"
    $resp = Invoke-RestMethod -Uri $url -Method Post -Body $body -ContentType "application/x-www-form-urlencoded"
    $Global:AccessToken = $resp.access_token
    $Global:TokenExpiry = (Get-Date).AddSeconds([int]$resp.expires_in - 300)
    return $Global:AccessToken
}

##### Function for obtaining access token via client secret

In [ ]:
$config = Get-Content "C:\vs\config\clientconfiguration.json" | ConvertFrom-Json
$Global:AccessToken = $null
$Global:TokenExpiry = Get-Date

function Get-AccessToken {
    if ($Global:AccessToken -and (Get-Date) -lt $Global:TokenExpiry) { return $Global:AccessToken }
    $body = @{
        client_id     = $config.ClientId
        client_secret = $config.ClientSecret
        scope         = "https://graph.microsoft.com/.default"
        grant_type    = "client_credentials"
    }
    $url = "https://login.microsoftonline.com/$($config.TenantId)/oauth2/v2.0/token"
    $resp = Invoke-RestMethod -Uri $url -Method Post -Body $body -ContentType "application/x-www-form-urlencoded"
    $Global:AccessToken = $resp.access_token
    $Global:TokenExpiry = (Get-Date).AddSeconds([int]$resp.expires_in - 300)
    return $Global:AccessToken
}

##### Direct REST call to endpoint via Invoke-RestMethod

In [ ]:
$AccessToken = Get-AccessToken
$allDevices = [System.Collections.Generic.List[object]]::new()
$url = "https://graph.microsoft.com/v1.0/devices"
$headers = @{ Authorization = "Bearer $(Get-AccessToken)" }

do {
    $response = Invoke-RestMethod -Uri $url -Headers $headers -Method Get
    $allDevices.AddRange($response.value)
    $url = $response.'@odata.nextLink'
} while ($url)

$allDevices | Format-Table displayName, operatingSystem, operatingSystemVersion

---
# SDK via cmdlet
> Using built-in SDK cmdlet (access tokens and pagination handled by module)

In [ ]:
$config = Get-Content "C:\vs\config\clientconfiguration.json" | ConvertFrom-Json
Connect-MgGraph -TenantId $config.TenantId -ClientId $config.ClientId -CertificateThumbprint $config.Thumbprint -NoWelcome

$allDevices = Get-MgDevice -All

$allDevices | Format-Table displayName, operatingSystem, operatingSystemVersion

Disconnect-MgGraph | Out-Null

---
# SDK via URL 
> Using Invoke-MgGraphRequest within SDK (access tokens handle by module, pagination is not)  
> Use case when there are NO built-in cmdlets for the target endpoint

In [ ]:
$config = Get-Content "C:\vs\config\clientconfiguration.json" | ConvertFrom-Json
Connect-MgGraph -TenantId $config.TenantId -ClientId $config.ClientId -CertificateThumbprint $config.Thumbprint -NoWelcome

$allDevices = [System.Collections.Generic.List[object]]::new()
$url = "https://graph.microsoft.com/v1.0/devices"

do {
    $json = Invoke-MgGraphRequest -Uri $url -Method GET -OutputType Json
    $response = $json | ConvertFrom-Json
    if ($response.value) {
        $allDevices.AddRange($response.value)
    }
    $url = $response.'@odata.nextLink'
}
while ($url)

$allDevices | Format-Table displayName, operatingSystem, operatingSystemVersion

Disconnect-MgGraph | Out-Null